# 💰 Personal Expense Tracker

> A complete, interactive expense management system built in Python.

---

## 📌 Overview

This project helps you **log**, **categorize**, and **manage** your daily expenses — all from a clean terminal menu. Your data is persisted in a CSV file so you never lose track of your spending.

### ✅ Features
| Feature | Description |
|---|---|
| ➕ Add Expense | Log a new expense with date, category, amount & description |
| 📋 View Expenses | Display all recorded expenses in a formatted table |
| 📊 Track Budget | Set a monthly budget and monitor remaining balance |
| 💾 Save / Load | Persist data to CSV; auto-loads on startup |
| 🖥️ Interactive Menu | Simple numbered menu for ease of use |

---

## 🗂️ Project Structure

```
personal_expense_tracker/
│
├── personal_expense_tracker.ipynb   ← Main notebook (this file)
└── expenses.csv                     ← Auto-generated data file
```

---

## 📦 Requirements

- Python 3.x (standard library only — no external packages needed!)
- Modules used: `csv`, `os`, `datetime`


---
## 🔧 Section 1: Imports & Configuration

In [ ]:
# ─────────────────────────────────────────────
#  Standard Library Imports
# ─────────────────────────────────────────────
import csv
import os
from datetime import datetime

# ─────────────────────────────────────────────
#  Global Configuration
# ─────────────────────────────────────────────
EXPENSE_FILE   = "expenses.csv"           # CSV file to persist expenses
CSV_FIELDNAMES = ["date", "category", "amount", "description"]  # Column order

VALID_CATEGORIES = [
    "Food", "Travel", "Shopping", "Health",
    "Entertainment", "Utilities", "Education", "Other"
]

# Runtime state
expenses: list[dict] = []   # In-memory list of expense dictionaries
monthly_budget: float = 0.0  # User-defined monthly budget

print("✅ Imports and configuration loaded successfully.")

---
## 🛠️ Section 2: Helper / Utility Functions

In [ ]:
def validate_date(date_str: str) -> bool:
    """
    Validate that a date string matches the YYYY-MM-DD format.

    Parameters
    ----------
    date_str : str
        The date string to validate.

    Returns
    -------
    bool
        True if the date is valid and correctly formatted, False otherwise.

    Examples
    --------
    >>> validate_date("2024-09-18")
    True
    >>> validate_date("18-09-2024")
    False
    """
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return True
    except ValueError:
        return False


def validate_amount(amount_str: str) -> bool:
    """
    Validate that the given string represents a positive numeric amount.

    Parameters
    ----------
    amount_str : str
        The amount string entered by the user.

    Returns
    -------
    bool
        True if the string can be converted to a positive float, False otherwise.

    Examples
    --------
    >>> validate_amount("15.50")
    True
    >>> validate_amount("-5")
    False
    """
    try:
        return float(amount_str) > 0
    except ValueError:
        return False


def validate_expense(expense: dict) -> bool:
    """
    Check that all required keys exist and are non-empty in an expense dictionary.

    Parameters
    ----------
    expense : dict
        A dictionary with keys: 'date', 'category', 'amount', 'description'.

    Returns
    -------
    bool
        True if all fields are present and non-empty, False otherwise.
    """
    required_fields = ["date", "category", "amount", "description"]
    return all(expense.get(field, "").strip() != "" for field in required_fields)


def print_separator(char: str = "─", width: int = 55) -> None:
    """Print a horizontal separator line for visual clarity."""
    print(char * width)


def get_total_expenses() -> float:
    """
    Calculate the sum of all recorded expense amounts.

    Returns
    -------
    float
        Total amount spent across all logged expenses.
    """
    return sum(float(e["amount"]) for e in expenses)


print("✅ Helper functions defined.")

---
## ➕ Section 3: Add Expense

In [ ]:
def add_expense() -> None:
    """
    Interactively prompt the user to enter expense details and store the record.

    The function collects:
        - date        : in YYYY-MM-DD format
        - category    : chosen from a predefined list
        - amount      : a positive float value
        - description : a brief text description

    The expense is appended to the global `expenses` list as a dictionary.

    Raises
    ------
    None — all validation is handled internally with re-prompts.
    """
    print_separator()
    print("  ➕  ADD NEW EXPENSE")
    print_separator()

    # ── DATE ──────────────────────────────────
    while True:
        date = input("  📅 Date (YYYY-MM-DD) [press Enter for today]: ").strip()
        if date == "":
            date = datetime.today().strftime("%Y-%m-%d")
            print(f"     Using today's date: {date}")
            break
        if validate_date(date):
            break
        print("  ❌ Invalid date format. Please use YYYY-MM-DD.")

    # ── CATEGORY ──────────────────────────────
    print("\n  🏷️  Available Categories:")
    for i, cat in enumerate(VALID_CATEGORIES, 1):
        print(f"      {i}. {cat}")

    while True:
        cat_input = input("\n  Choose category number (or type custom): ").strip()
        if cat_input.isdigit() and 1 <= int(cat_input) <= len(VALID_CATEGORIES):
            category = VALID_CATEGORIES[int(cat_input) - 1]
            break
        elif cat_input != "":
            category = cat_input.capitalize()
            break
        print("  ❌ Please select a valid category.")

    # ── AMOUNT ────────────────────────────────
    while True:
        amount_str = input("  💵 Amount (e.g. 15.50): ").strip()
        if validate_amount(amount_str):
            amount = float(amount_str)
            break
        print("  ❌ Please enter a valid positive number.")

    # ── DESCRIPTION ───────────────────────────
    while True:
        description = input("  📝 Description: ").strip()
        if description:
            break
        print("  ❌ Description cannot be empty.")

    # ── STORE ─────────────────────────────────
    expense = {
        "date"        : date,
        "category"    : category,
        "amount"      : amount,
        "description" : description,
    }
    expenses.append(expense)

    print_separator()
    print(f"  ✅ Expense added: {category} — ₹{amount:.2f} on {date}")
    print_separator()


print("✅ add_expense() defined.")

---
## 📋 Section 4: View Expenses

In [ ]:
def view_expenses() -> None:
    """
    Display all recorded expenses in a formatted table.

    Skips any entries that fail validation (missing or empty required fields)
    and notifies the user about such incomplete records.

    The table shows:
        - Row number
        - Date
        - Category
        - Amount (formatted to 2 decimal places)
        - Description

    A grand total is displayed at the bottom.
    """
    print_separator()
    print("  📋  ALL EXPENSES")
    print_separator()

    if not expenses:
        print("  ℹ️  No expenses recorded yet. Add one first!")
        print_separator()
        return

    # Table header
    print(f"  {'#':<4} {'Date':<13} {'Category':<16} {'Amount':>10}  Description")
    print_separator("·")

    valid_count   = 0
    skipped_count = 0
    total         = 0.0

    for idx, expense in enumerate(expenses, 1):
        if not validate_expense(
            {k: str(v) for k, v in expense.items()}
        ):
            print(f"  ⚠️  Row {idx}: Incomplete data — skipped.")
            skipped_count += 1
            continue

        amount = float(expense["amount"])
        total += amount
        valid_count += 1

        print(
            f"  {idx:<4} "
            f"{expense['date']:<13} "
            f"{expense['category']:<16} "
            f"₹{amount:>8.2f}  "
            f"{expense['description']}"
        )

    # Summary footer
    print_separator("·")
    print(f"  {'TOTAL':<34} ₹{total:>8.2f}")
    print_separator()

    if skipped_count:
        print(f"  ⚠️  {skipped_count} incomplete record(s) were skipped.")

    print(f"  📌 Showing {valid_count} valid expense(s).")
    print_separator()


print("✅ view_expenses() defined.")

---
## 📊 Section 5: Budget Tracking

In [ ]:
def set_budget() -> None:
    """
    Prompt the user to set or update their monthly budget.

    Updates the global `monthly_budget` variable. The user must enter a
    positive numeric value; re-prompts on invalid input.
    """
    global monthly_budget

    print_separator()
    print("  💳  SET MONTHLY BUDGET")
    print_separator()

    while True:
        budget_str = input("  Enter your monthly budget amount: ₹").strip()
        if validate_amount(budget_str):
            monthly_budget = float(budget_str)
            print(f"  ✅ Monthly budget set to ₹{monthly_budget:.2f}")
            print_separator()
            return
        print("  ❌ Please enter a valid positive number.")


def track_budget() -> None:
    """
    Compare total expenses against the monthly budget and display a status report.

    Behaviour:
        - If no budget has been set, prompts the user to set one first.
        - Shows total spent, budget, and remaining/overspent amount.
        - Displays a warning if the budget has been exceeded.
        - Displays a progress bar indicating budget utilisation (0–100%).
    """
    print_separator()
    print("  📊  BUDGET TRACKER")
    print_separator()

    if monthly_budget <= 0:
        print("  ⚠️  No budget set yet. Please set a budget first.")
        print_separator()
        set_budget()
        if monthly_budget <= 0:
            return

    total_spent = get_total_expenses()
    remaining   = monthly_budget - total_spent
    used_pct    = min((total_spent / monthly_budget) * 100, 100)

    # ASCII progress bar
    bar_width  = 30
    filled     = int(bar_width * used_pct / 100)
    bar        = "█" * filled + "░" * (bar_width - filled)

    print(f"  Monthly Budget  : ₹{monthly_budget:>10.2f}")
    print(f"  Total Spent     : ₹{total_spent:>10.2f}")
    print_separator("·")
    print(f"  [{bar}] {used_pct:.1f}%")
    print_separator("·")

    if remaining < 0:
        print(f"  🚨 You have EXCEEDED your budget by ₹{abs(remaining):.2f}!")
    elif remaining == 0:
        print("  ⚠️  You have used exactly your entire budget.")
    else:
        print(f"  ✅ You have ₹{remaining:.2f} remaining for the month.")

    print_separator()


print("✅ set_budget() and track_budget() defined.")

---
## 💾 Section 6: Save & Load (File Handling)

In [ ]:
def save_expenses() -> None:
    """
    Save all in-memory expenses to a CSV file.

    Each row in the CSV contains: date, category, amount, description.
    The file is overwritten on each save to reflect the current state.
    If no expenses exist, a notification is displayed and no file is written.

    File written to: `EXPENSE_FILE` (default: 'expenses.csv')
    """
    print_separator()
    print("  💾  SAVE EXPENSES")
    print_separator()

    if not expenses:
        print("  ℹ️  Nothing to save — expense list is empty.")
        print_separator()
        return

    try:
        with open(EXPENSE_FILE, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES)
            writer.writeheader()
            writer.writerows(expenses)

        print(f"  ✅ {len(expenses)} expense(s) saved to '{EXPENSE_FILE}'.")

    except IOError as e:
        print(f"  ❌ Error saving file: {e}")

    print_separator()


def load_expenses() -> None:
    """
    Load expenses from the CSV file into the in-memory `expenses` list.

    Called automatically when the program starts. If the file does not exist,
    the function exits silently (fresh start). Each row is validated before
    loading; invalid rows are skipped with a warning.

    The 'amount' field is coerced from string to float during loading.

    File read from: `EXPENSE_FILE` (default: 'expenses.csv')
    """
    global expenses

    if not os.path.exists(EXPENSE_FILE):
        print("  ℹ️  No saved data found. Starting fresh.")
        return

    loaded      = []
    skipped     = 0

    try:
        with open(EXPENSE_FILE, mode="r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if validate_expense(row):
                    row["amount"] = float(row["amount"])  # Cast to float
                    loaded.append(row)
                else:
                    skipped += 1

        expenses = loaded
        print(f"  ✅ Loaded {len(expenses)} expense(s) from '{EXPENSE_FILE}'.")

        if skipped:
            print(f"  ⚠️  {skipped} invalid row(s) were skipped during load.")

    except IOError as e:
        print(f"  ❌ Error reading file: {e}")


print("✅ save_expenses() and load_expenses() defined.")

---
## 🖥️ Section 7: Interactive Menu

In [ ]:
def display_menu() -> None:
    """
    Print the main menu options to the console.

    This is a pure display function — it does not accept input.
    """
    print_separator("═")
    print("  💰  PERSONAL EXPENSE TRACKER")
    print_separator("═")
    print("  1. ➕  Add Expense")
    print("  2. 📋  View Expenses")
    print("  3. 📊  Track Budget")
    print("  4. 💳  Set Monthly Budget")
    print("  5. 💾  Save Expenses")
    print("  6. 🚪  Exit")
    print_separator("═")


def run_tracker() -> None:
    """
    Main entry point: launch the interactive expense tracker.

    Workflow:
        1. Load any previously saved expenses from disk.
        2. Display the menu in a loop.
        3. Route user input to the appropriate function.
        4. On exit (option 6), auto-save and quit gracefully.

    Menu Options:
        1 → add_expense()
        2 → view_expenses()
        3 → track_budget()
        4 → set_budget()
        5 → save_expenses()
        6 → save_expenses() + exit
    """
    print("\n" + "═" * 55)
    print("  Welcome to Personal Expense Tracker 💰")
    print("═" * 55)

    # Auto-load saved expenses on startup
    load_expenses()

    while True:
        print()
        display_menu()

        choice = input("  👉 Enter your choice (1–6): ").strip()

        print()

        if choice == "1":
            add_expense()

        elif choice == "2":
            view_expenses()

        elif choice == "3":
            track_budget()

        elif choice == "4":
            set_budget()

        elif choice == "5":
            save_expenses()

        elif choice == "6":
            print_separator("═")
            print("  💾 Auto-saving before exit...")
            save_expenses()
            print("  👋 Thank you for using Expense Tracker. Goodbye!")
            print_separator("═")
            break

        else:
            print("  ❌ Invalid choice. Please enter a number between 1 and 6.")


print("✅ display_menu() and run_tracker() defined.")

---
## 🧪 Section 8: Unit Tests

Quick sanity checks for all core utility functions — no external libraries needed.

In [ ]:
def run_tests() -> None:
    """
    Run a suite of unit tests for helper and core functions.

    Tests cover:
        - Date format validation (valid, invalid, empty)
        - Amount validation (positive float, negative, non-numeric)
        - Expense record validation (complete, missing fields)
        - Total expense calculation
    """
    print("═" * 55)
    print("  🧪  RUNNING UNIT TESTS")
    print("═" * 55)

    passed = 0
    failed = 0

    def assert_test(description: str, result: bool, expected: bool) -> None:
        nonlocal passed, failed
        status = "✅ PASS" if result == expected else "❌ FAIL"
        if result == expected:
            passed += 1
        else:
            failed += 1
        print(f"  {status}  |  {description}")

    # ── validate_date ──────────────────────────
    print("\n  📅 validate_date()")
    assert_test("Valid date '2024-09-18'",          validate_date("2024-09-18"),  True)
    assert_test("Wrong format '18-09-2024'",        validate_date("18-09-2024"), False)
    assert_test("Non-date string 'hello'",          validate_date("hello"),      False)
    assert_test("Empty string ''",                  validate_date(""),           False)
    assert_test("Today's date (auto-generated)",
                validate_date(datetime.today().strftime("%Y-%m-%d")), True)

    # ── validate_amount ────────────────────────
    print("\n  💵 validate_amount()")
    assert_test("Valid float '15.50'",              validate_amount("15.50"),  True)
    assert_test("Valid integer '100'",              validate_amount("100"),    True)
    assert_test("Negative amount '-5'",             validate_amount("-5"),     False)
    assert_test("Zero amount '0'",                  validate_amount("0"),      False)
    assert_test("Non-numeric 'abc'",                validate_amount("abc"),    False)

    # ── validate_expense ───────────────────────
    print("\n  📋 validate_expense()")
    good_expense = {"date": "2024-09-18", "category": "Food",
                    "amount": "15.50",    "description": "Lunch"}
    missing_desc = {"date": "2024-09-18", "category": "Food",
                    "amount": "15.50",    "description": ""}
    missing_key  = {"date": "2024-09-18", "category": "Food", "amount": "15.50"}

    assert_test("Complete valid expense",           validate_expense(good_expense), True)
    assert_test("Empty description field",          validate_expense(missing_desc), False)
    assert_test("Missing description key",          validate_expense(missing_key),  False)

    # ── get_total_expenses ─────────────────────
    print("\n  ➕ get_total_expenses()")
    global expenses
    original = expenses.copy()

    expenses = [
        {"date": "2024-01-01", "category": "Food",   "amount": 20.0,  "description": "A"},
        {"date": "2024-01-02", "category": "Travel",  "amount": 30.5,  "description": "B"},
        {"date": "2024-01-03", "category": "Health",  "amount": 49.5,  "description": "C"},
    ]
    assert_test("Sum of [20, 30.5, 49.5] == 100.0",
                get_total_expenses() == 100.0, True)
    expenses = []
    assert_test("Empty list → total == 0.0",
                get_total_expenses() == 0.0, True)

    expenses = original  # Restore state

    # ── Summary ────────────────────────────────
    print("\n" + "═" * 55)
    print(f"  Results: {passed} passed | {failed} failed | {passed + failed} total")
    print("═" * 55)


run_tests()

---
## 🚀 Section 9: Run the Tracker

> **Run the cell below to launch the interactive menu.**  
> Type a number and press **Enter** to navigate. Your data is auto-saved on exit.

In [ ]:
# ─────────────────────────────────────────────
#  Launch the Expense Tracker
#  (Re-run this cell any time to restart)
# ─────────────────────────────────────────────
run_tracker()

---

## 📸 Sample Output

```
═══════════════════════════════════════════════════════
  Welcome to Personal Expense Tracker 💰
═══════════════════════════════════════════════════════
  ✅ Loaded 3 expense(s) from 'expenses.csv'.

  ═══════════════════════════════════════════════════
    💰  PERSONAL EXPENSE TRACKER
  ═══════════════════════════════════════════════════
    1. ➕  Add Expense
    2. 📋  View Expenses
    3. 📊  Track Budget
    4. 💳  Set Monthly Budget
    5. 💾  Save Expenses
    6. 🚪  Exit
  ═══════════════════════════════════════════════════
  👉 Enter your choice (1–6): 3

  ───────────────────────────────────────────────────
    📊  BUDGET TRACKER
  ───────────────────────────────────────────────────
    Monthly Budget  : ₹   5000.00
    Total Spent     : ₹   1250.00
  ···················································
    [███████░░░░░░░░░░░░░░░░░░░░░░░] 25.0%
  ···················································
    ✅ You have ₹3750.00 remaining for the month.
  ───────────────────────────────────────────────────
```

---

## 👤 Author

| Field | Info |
|---|---|
| **Project** | Personal Expense Tracker |
| **Course** | Simplilearn — Python Course-End Project |
| **Language** | Python 3.x |
| **Libraries** | `csv`, `os`, `datetime` (standard library only) |

---

*Feel free to fork, star ⭐, and build on top of this!*
